# 06. Perbandingan Model (Random Forest vs XGBoost, dengan dan tanpa CSL)

Notebook ini membandingkan empat model pada dataset **Diabetes Health Indicators (BRFSS 2015)**: Random Forest dan XGBoost, masing-masing tanpa dan dengan **cost-sensitive learning (CSL)**.

Metrik yang dilaporkan: recall per kelas, Macro F1, G-Mean, dan Accuracy.

Isi dan logika perbandingan mengikuti script sumber `compare_models.py`.

## 0. Mengatur Folder Kerja

Notebook ini disimpan di folder `notebooks`, sedangkan data dan artifacts berada di folder `glysense_backend`. Sel berikut memindahkan folder kerja satu tingkat ke atas agar jalur seperti `data/...` dan `artifacts/...` ditemukan. Aman dijalankan berulang.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print(os.getcwd())

d:\Tugas Akhir\Aplikasi Diabetes\glysense_backend


## 1. Membaca Dataset

Jalur file mengasumsikan notebook dijalankan dari direktori utama `glysense_backend`.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    f1_score,
    accuracy_score
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

file_path = "data/diabetes_012_health_indicators_BRFSS2015.csv"

df = pd.read_csv(file_path)

## 2. Memisahkan Fitur dan Target

In [3]:
X = df.drop(columns=["Diabetes_012"])
y = df["Diabetes_012"].astype(int)

## 3. Train-Test Split

Catatan: pembagian ini menggunakan `random_state=42` sesuai script asli, sehingga **berbeda** dari pembagian final penelitian (`random_state=2026`). Notebook ini bersifat eksperimen awal dan tidak menyimpan file apa pun.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

## 4. Fungsi G-Mean Multiclass

G-Mean dihitung sebagai akar pangkat tiga dari hasil kali recall ketiga kelas.

In [5]:
def multiclass_gmean(y_true, y_pred):
    recalls = recall_score(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        average=None,
        zero_division=0
    )

    gmean = np.prod(recalls) ** (1 / len(recalls))

    return recalls, gmean

## 5. Fungsi Evaluasi Model

In [6]:
def evaluate_model(model_name, y_true, y_pred):
    recalls, gmean = multiclass_gmean(
        y_true,
        y_pred
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    print("\n" + "=" * 75)
    print(model_name)
    print("=" * 75)

    print("\nRecall per kelas:")
    print(f"Kelas 0 - Tidak Diabetes : {recalls[0]:.4f}")
    print(f"Kelas 1 - Prediabetes    : {recalls[1]:.4f}")
    print(f"Kelas 2 - Diabetes       : {recalls[2]:.4f}")

    print(f"\nMacro F1 : {macro_f1:.4f}")
    print(f"G-Mean   : {gmean:.4f}")
    print(f"Accuracy : {accuracy:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "Model": model_name,
        "Recall_0": recalls[0],
        "Recall_1": recalls[1],
        "Recall_2": recalls[2],
        "Macro_F1": macro_f1,
        "G_Mean": gmean,
        "Accuracy": accuracy
    }


results = []

## 6. Random Forest Tanpa CSL

In [7]:
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_baseline.fit(
    X_train,
    y_train
)

rf_baseline_pred = rf_baseline.predict(
    X_test
)

results.append(
    evaluate_model(
        "Random Forest Tanpa CSL",
        y_test,
        rf_baseline_pred
    )
)


Random Forest Tanpa CSL

Recall per kelas:
Kelas 0 - Tidak Diabetes : 0.9676
Kelas 1 - Prediabetes    : 0.0000
Kelas 2 - Diabetes       : 0.1971

Macro F1 : 0.3982
G-Mean   : 0.0000
Accuracy : 0.8426

Confusion Matrix:
[[41358    44  1339]
 [  828     0    98]
 [ 5669     7  1393]]


## 7. Random Forest Dengan CSL

In [8]:
rf_csl = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_csl.fit(
    X_train,
    y_train
)

rf_csl_pred = rf_csl.predict(
    X_test
)

results.append(
    evaluate_model(
        "Random Forest Dengan CSL",
        y_test,
        rf_csl_pred
    )
)


Random Forest Dengan CSL

Recall per kelas:
Kelas 0 - Tidak Diabetes : 0.8486
Kelas 1 - Prediabetes    : 0.0011
Kelas 2 - Diabetes       : 0.5424

Macro F1 : 0.4388
G-Mean   : 0.0792
Accuracy : 0.7905

Confusion Matrix:
[[36271   248  6222]
 [  546     1   379]
 [ 3186    49  3834]]


## 8. XGBoost Tanpa CSL

In [9]:
xgb_baseline = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_baseline.fit(
    X_train,
    y_train
)

xgb_baseline_pred = xgb_baseline.predict(
    X_test
)

results.append(
    evaluate_model(
        "XGBoost Tanpa CSL",
        y_test,
        xgb_baseline_pred
    )
)


XGBoost Tanpa CSL

Recall per kelas:
Kelas 0 - Tidak Diabetes : 0.9787
Kelas 1 - Prediabetes    : 0.0000
Kelas 2 - Diabetes       : 0.1828

Macro F1 : 0.3978
G-Mean   : 0.0000
Accuracy : 0.8499

Confusion Matrix:
[[41831     0   910]
 [  830     0    96]
 [ 5777     0  1292]]


## 9. XGBoost Dengan CSL

In [10]:
sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_csl = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

xgb_csl.fit(
    X_train,
    y_train,
    sample_weight=sample_weights
)

xgb_csl_pred = xgb_csl.predict(
    X_test
)

results.append(
    evaluate_model(
        "XGBoost Dengan CSL",
        y_test,
        xgb_csl_pred
    )
)


XGBoost Dengan CSL

Recall per kelas:
Kelas 0 - Tidak Diabetes : 0.6404
Kelas 1 - Prediabetes    : 0.2948
Kelas 2 - Diabetes       : 0.6186

Macro F1 : 0.4204
G-Mean   : 0.4888
Accuracy : 0.6311

Confusion Matrix:
[[27373  7408  7960]
 [  235   273   418]
 [ 1057  1639  4373]]


## 10. Tabel Perbandingan

In [11]:
results_df = pd.DataFrame(results)

print("=" * 100)
print("RINGKASAN PERBANDINGAN MODEL")
print("=" * 100)

print(
    results_df.round(4).to_string(
        index=False
    )
)

RINGKASAN PERBANDINGAN MODEL
                   Model  Recall_0  Recall_1  Recall_2  Macro_F1  G_Mean  Accuracy
 Random Forest Tanpa CSL    0.9676    0.0000    0.1971    0.3982  0.0000    0.8426
Random Forest Dengan CSL    0.8486    0.0011    0.5424    0.4388  0.0792    0.7905
       XGBoost Tanpa CSL    0.9787    0.0000    0.1828    0.3978  0.0000    0.8499
      XGBoost Dengan CSL    0.6404    0.2948    0.6186    0.4204  0.4888    0.6311
